# Activité Pratique N°3 — RAG Multi Modal Agentique
## Étape 3 — Indexation et stockage dans une base de données vectorielle (ChromaDB)

**IA Générative Multi Modale — 5IAD G1, EMSI** · Étudiant : BENHAYOUN SADAFI Mohamed

**Objectif de cette étape :** stocker les embeddings des textes et des images dans ChromaDB avec leurs métadonnées, et gérer les réindexations grâce à un identifiant SHA-256 du document.

Document source : *Attention Is All You Need* (`pdfs/attention is all you need.pdf`).

> 🔁 **Code refait** — la configuration, le chargement de CLIP et les fonctions d'encodage, repris de l'étape 2. Chaque notebook est indépendant et recrée ce dont il a besoin.

In [1]:
import base64
import hashlib
import json
import os
import re
from io import BytesIO
from pathlib import Path

from IPython.display import Markdown, display
from PIL import Image

RACINE = Path.cwd()
PDF_PATH = RACINE / "pdfs" / "attention is all you need.pdf"
OUTPUT_DIR = RACINE / "resultats_pdf"
TEXT_DIR = OUTPUT_DIR / "textes"
IMAGE_DIR = OUTPUT_DIR / "images"
CHROMA_DIR = RACINE / "chroma_db"
COLLECTION_NOM = "pdf_clip_vit_b32"
IMAGE_REQUETE = RACINE / "images_requetes" / "architecture_transformer.png"
print("PDF :", PDF_PATH.name, "| existe :", PDF_PATH.is_file())

PDF : attention is all you need.pdf | existe : True


In [2]:
import open_clip
import torch
from open_clip.tokenizer import SimpleTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
bpe = SimpleTokenizer()


@torch.inference_mode()
def encoder_textes(textes):
    vecteurs = model.encode_text(tokenizer(textes).to(DEVICE)).float()
    vecteurs = vecteurs / vecteurs.norm(dim=-1, keepdim=True).clamp_min(1e-12)   # normalisation L2
    return vecteurs.cpu().tolist()


@torch.inference_mode()
def encoder_image(image):
    # `image` : chemin de fichier ou image PIL
    if isinstance(image, (str, Path)):
        with Image.open(image) as img:
            tenseur = preprocess(img.convert("RGB"))
    else:
        tenseur = preprocess(image.convert("RGB"))
    vecteur = model.encode_image(tenseur.unsqueeze(0).to(DEVICE)).float()
    vecteur = vecteur / vecteur.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return vecteur.cpu().tolist()[0]


print("CLIP", MODEL_NAME, PRETRAINED, "chargé sur", DEVICE)

C:\Users\user\Desktop\5IAD G1 BENHAYOUN SADAFI Mohamed, emsi, matiere ia multimodale\Activité Pratique N°3 - RAG Multi Modal Agentique\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CLIP ViT-B-32 laion2b_s34b_b79k chargé sur cuda


> 🔁 **Code refait** — la fonction de découpage `decouper_texte`, repris de l'étape 1. Chaque notebook est indépendant et recrée ce dont il a besoin.

In [3]:
def decouper_texte(texte, taille=60, chevauchement=10):
    tokens = bpe.encode(texte)
    for debut in range(0, len(tokens), taille - chevauchement):
        fragment = bpe.decode(tokens[debut:debut + taille]).strip()
        if fragment:
            yield fragment
        if debut + taille >= len(tokens):
            break


def numero_page(chemin):
    correspondance = re.search(r"page_(\d+)", Path(chemin).stem)
    return int(correspondance.group(1)) if correspondance else -1

### Initialiser la base vectorielle

Base **persistante sur disque** (`chroma_db/`), espace **cosinus**. `embedding_function=None` : les embeddings sont calculés par notre code (CLIP), pas par ChromaDB.

In [4]:
import chromadb

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name=COLLECTION_NOM, embedding_function=None,
                                             metadata={"hnsw:space": "cosine"})
print("Collection :", collection.name, "| éléments déjà présents :", collection.count())

Collection : pdf_clip_vit_b32 | éléments déjà présents : 0


### Identifiant du document (SHA-256)

L'identifiant est calculé à partir du **contenu** du PDF : si on relance l'indexation, on supprime d'abord tout ce qui porte cet identifiant, ce qui évite les doublons et les fragments obsolètes.

In [5]:
document_id = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()[:24]
print("document_id :", document_id)
avant = collection.count()
collection.delete(where={"document_id": document_id})
print(f"Éléments supprimés (ancienne indexation de ce PDF) : {avant - collection.count()}")

document_id : d87d482d5ae7960e2e43d7dd
Éléments supprimés (ancienne indexation de ce PDF) : 0


### Indexer les fragments de texte (par lots de 64)

In [6]:
ids, documents, metadatas = [], [], []
for fichier in sorted(TEXT_DIR.glob("page_*.txt")):
    page = numero_page(fichier)
    for index, fragment in enumerate(decouper_texte(fichier.read_text(encoding="utf-8"))):
        ids.append(f"{document_id}:texte:{page}:{index}")
        documents.append(fragment)
        metadatas.append({"type": "texte", "document_id": document_id, "pdf": PDF_PATH.name,
                          "page": page, "fragment": index, "source": str(fichier)})

TAILLE_LOT = 64
for debut in range(0, len(documents), TAILLE_LOT):
    fin = debut + TAILLE_LOT
    collection.upsert(ids=ids[debut:fin], documents=documents[debut:fin], metadatas=metadatas[debut:fin],
                      embeddings=encoder_textes(documents[debut:fin]))
print(f"{len(documents)} fragments de texte indexés.")

171 fragments de texte indexés.


### Indexer les images dans la même collection

Le champ `documents` d'une image n'est qu'une description technique : son embedding est calculé à partir des **pixels**.

In [7]:
extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff", ".jp2"}
nombre_images_indexees = 0
for fichier in sorted(IMAGE_DIR.iterdir()):
    if fichier.suffix.lower() not in extensions:
        continue
    collection.upsert(ids=[f"{document_id}:image:{fichier.name}"], embeddings=[encoder_image(fichier)],
                      documents=[f"Image extraite du PDF : {fichier.name}"],
                      metadatas=[{"type": "image", "document_id": document_id, "pdf": PDF_PATH.name,
                                  "page": numero_page(fichier), "source": str(fichier)}])
    nombre_images_indexees += 1
print(f"{nombre_images_indexees} images indexées.")
print(f"Total dans ChromaDB : {collection.count()} éléments.")

3 images indexées.
Total dans ChromaDB : 174 éléments.


### Vérifier le contenu stocké

In [8]:
echantillon = collection.get(limit=2, include=["documents", "metadatas", "embeddings"])
for ident, doc, meta, emb in zip(echantillon["ids"], echantillon["documents"], echantillon["metadatas"],
                                 echantillon["embeddings"]):
    print(ident)
    print("  métadonnées :", {k: v for k, v in meta.items() if k != "source"})
    print("  document    :", doc[:100], "…")
    print("  embedding   :", len(emb), "valeurs\n")

images = collection.get(where={"type": "image"}, include=["metadatas"])
print("Images stockées :", [(m["page"], Path(m["source"]).name) for m in images["metadatas"]])

d87d482d5ae7960e2e43d7dd:texte:1:0
  métadonnées : {'page': 1, 'fragment': 0, 'document_id': 'd87d482d5ae7960e2e43d7dd', 'pdf': 'attention is all you need.pdf', 'type': 'texte'}
  document    : attention is all you need ashish vaswani ∗ noam shazeer ∗ niki parmar ∗ jakob uszkoreit ∗ google bra …
  embedding   : 512 valeurs

d87d482d5ae7960e2e43d7dd:texte:1:1
  métadonnées : {'fragment': 1, 'page': 1, 'pdf': 'attention is all you need.pdf', 'type': 'texte', 'document_id': 'd87d482d5ae7960e2e43d7dd'}
  document    : kip @ google . com usz @ google . com llion jones ∗ aidan n . gomez ∗† łukasz kaiser ∗ google resear …
  embedding   : 512 valeurs

Images stockées : [(3, 'page_003_image_94.png'), (4, 'page_004_image_102.png'), (4, 'page_004_image_104.png')]
